> **History.** Sutskever, Vinyals, and Le (Google, 2014) showed that a fixed-size vector could represent an entire sentence well enough for machine translation. Bahdanau et al. (2015) introduced additive attention: instead of one fixed vector, the decoder could read a weighted blend of all encoder hidden states. This was the direct ancestor of cross-attention. The encoder-decoder pattern still powers T5, BART, mT5, and every seq2seq production model today.
>
> **Where you are.** You've built a full Transformer in `02-transformers/` — you understand self-attention, positional encoding, residuals, and multi-head attention. The gap: you built decoder-only (MiniLM) and saw encoder-only briefly. You have not yet trained the encoder-decoder architecture.
>
> **Notation.** $S$ — source sequence length; $T$ — target sequence length; $Q$ from decoder, $K/V$ from encoder in cross-attention; `mask=None` for encoder (bidirectional), causal mask for decoder; teacher forcing: feed ground-truth target at each decoding step during training.

---

## Prerequisite Bridge — From `02-transformers/transformers.ipynb`

| Foundation | Role in this notebook |
|---|---|
| Multi-head self-attention (Q, K, V, scaled dot-product) | Used directly in both encoder and decoder blocks |
| Causal attention mask (triangular) | The decoder's causal self-attention; the encoder deliberately removes it |
| Sinusoidal positional encoding | Carried forward unchanged into encoder and decoder embeddings |
| Residual connections + LayerNorm (Pre-LN) | Both blocks follow the same Pre-LN pattern |
| Decoder-only architecture (MiniLM) | This notebook adds cross-attention as a second sub-layer in the decoder |

> **If you haven't run `02-transformers/transformers.ipynb`** the attention mechanism, causal masking, and Pre-LN architecture used here will not be familiar — those derivations are not repeated.

## 0 · The Challenge

> **The mission**: Build an encoder-decoder Transformer from scratch and prove it learns a non-trivial mapping — integer sequence reversal (`[3, 1, 4, 1] → [1, 4, 1, 3]`).

**What we know so far:**
- Decoder-only Transformers (MiniLM) generate text left-to-right from context.
- **But we still can't**: map a variable-length source sequence to a different-length target, or give the decoder access to the full source representation at every decoding step.

**What this chapter unlocks:** Cross-attention — the decoder queries the encoder's full output at every step. The **anti-diagonal** pattern in the trained cross-attention map will prove the model learned the reversal.

## Encoder-Decoder Transformers in TensorFlow/Keras

| Step | Concept | Key Idea |
| ---- | -------- | --------- |
| 1 | The Contract | What encoder-decoder solves |
| 2 | The Encoder | Bidirectional attention — mask=None |
| 3 | The Bottleneck | Why naive concat fails |
| 4 | Cross-Attention | Q = decoder, K/V = encoder |
| 5 | Full Model + Training | `tf.GradientTape` seq2seq loop |
| 6 | Cross-Attention Map | Proves anti-diagonal routing |
| 6a | Free-Running Decoding | Teacher forcing vs. autoregressive greedy |
| 7 | Toy to Real | T5/BART parameter mapping |

In [ ]:
#  Setup: install + imports
import subprocess, sys
required = [
    ("numpy","numpy"), ("matplotlib","matplotlib"),
    ("tensorflow","tensorflow"), ("seaborn","seaborn"),
    ("plotly","plotly"), ("transformers","transformers"),
]
for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  ok  {pkg}")
    except ImportError:
        print(f"  installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  done {pkg}")

In [ ]:
#  Imports and reproducibility seed
import math, random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import seaborn as sns

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

print("TensorFlow version:", tf.__version__)
print("Seed fixed at 42 — every cell in this notebook is deterministic.")

---

## Part 1 — The Encoder-Decoder Contract

### What problem does it solve that decoder-only cannot?

A decoder-only model (GPT-style) produces one token at a time, conditioned on every token that came before. For **sequence reversal** — input `[3, 1, 4, 1]` must produce `[1, 4, 1, 3]` — the first output token (`1`) depends on the _last_ input token. A causal decoder at generation step 0 cannot look forward to position 3.

**The encoder-decoder contract:**
- The **encoder** reads the full source sequence bidirectionally and produces one enriched context vector per source token. It does not generate; it _enriches_.
- The **decoder** generates the target autoregressively, querying the full source map via cross-attention at every step.

$$\text{Cross-Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^{\top}}{\sqrt{d_k}}\right) V$$

where $Q$ comes from the **decoder** and $K, V$ come from the **encoder** output.

In [ ]:
#  Running example: integer sequence reversal
PAD, BOS, EOS = 10, 11, 12
VOCAB_SIZE = 13  # 0-9 digits + PAD + BOS + EOS
SEQ_LEN = 4     # source / target sequence length
D_MODEL = 32    # embedding / hidden dimension
N_HEADS = 4
D_FF = 64
N_LAYERS = 2

def make_reversal_pairs(n_samples, seq_len=SEQ_LEN, seed=42):
    rng = np.random.default_rng(seed)
    srcs, tgts = [], []
    for _ in range(n_samples):
        src = list(rng.integers(0, 10, size=seq_len))
        tgt = src[::-1]
        srcs.append(src); tgts.append(tgt)
    return srcs, tgts

train_srcs, train_tgts = make_reversal_pairs(2000)
val_srcs, val_tgts = make_reversal_pairs(200, seed=99)

print("Reversal task examples (first 5):")
print(f"  {'Source':<20}  Target (reversed)")
print(f"  {'-'*18}  {'-'*18}")
for s, t in zip(train_srcs[:5], train_tgts[:5]):
    print(f"  {str(s):<20}  {str(t)}")
print()
print(f"Training pairs : {len(train_srcs)}")
print(f"Vocab size     : {VOCAB_SIZE}  (0-9=digits, 10=PAD, 11=BOS, 12=EOS)")
print("Decoder input  : [BOS] + target[:-1]  (teacher forcing)")
print("Decoder target : target + [EOS]")

---

## Part 2 — The Encoder: Enriching Source Representations

### Bidirectional self-attention

In a **decoder** block, position $i$ can attend only to positions $0, \ldots, i$ — enforced by adding $-\infty$ to the upper-triangle before the softmax. In an **encoder** block, we simply pass `mask=None`. Every token sees every other token — backward _and_ forward.

```
decoder block:  attention_mask = causal_mask   # upper-triangle -> -inf
encoder block:  attention_mask = None          # nothing blocked
```

Everything else — `MultiHeadAttention`, `LayerNorm`, `FeedForward` — is identical.

In [ ]:
#  Shared attention primitive — reused by encoder and decoder
def sinusoidal_pe(max_seq: int, d_model: int) -> tf.Tensor:
    """Sinusoidal positional encoding: (max_seq, d_model)."""
    positions = tf.cast(tf.range(max_seq)[:, tf.newaxis], tf.float32)
    dims = tf.cast(tf.range(0, d_model, 2), tf.float32)
    freqs = 1.0 / tf.pow(10000.0, dims / tf.cast(d_model, tf.float32))
    angles = positions * freqs
    sin_part = tf.math.sin(angles)
    cos_part = tf.math.cos(angles)
    pe = tf.reshape(tf.stack([sin_part, cos_part], axis=2), [max_seq, d_model])
    return pe

class FeedForward(tf.keras.layers.Layer):
    def __init__(self, d_model, d_ff, **kwargs):
        super().__init__(**kwargs)
        self.dense1 = layers.Dense(d_ff, activation='relu')
        self.dense2 = layers.Dense(d_model)
    def call(self, x, training=False):
        return self.dense2(self.dense1(x))

class TransformerBlock(tf.keras.layers.Layer):
    """Pre-LN block: LN->MHA->residual, LN->FFN->residual.
    Pass mask=None for encoder (bidirectional), causal_mask for decoder.
    """
    def __init__(self, d_model, n_heads, d_ff, **kwargs):
        super().__init__(**kwargs)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.mha   = layers.MultiHeadAttention(num_heads=n_heads,
                                               key_dim=d_model // n_heads,
                                               dropout=0.0)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn   = FeedForward(d_model, d_ff)
    def call(self, x, mask=None, training=False):
        normed = self.norm1(x)
        attn_out = self.mha(normed, normed, attention_mask=mask, training=training)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x), training=training)
        return x

In [ ]:
#  MiniEncoder: embedding + PE + stack of bidirectional blocks
class MiniEncoder(tf.keras.layers.Layer):
    """
    Bidirectional encoder (BERT / T5-encoder style).
    Output: one enriched d_model-dimensional vector per source token.
    NOT a next-token predictor — a context enricher.
    """
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq=64, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = layers.Embedding(vocab_size, d_model, mask_zero=False)
        self.pe = sinusoidal_pe(max_seq, d_model)   # fixed, not trainable
        self.blocks = [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        self.norm_out = layers.LayerNormalization(epsilon=1e-6)

    def call(self, token_ids, training=False):
        S = tf.shape(token_ids)[1]
        x = self.token_emb(token_ids) + self.pe[:S]   # (B, S, d_model)
        for block in self.blocks:
            x = block(x, mask=None, training=training)  # <- mask=None is the key
        return self.norm_out(x)   # (B, S, d_model)

# Sanity check
tf.random.set_seed(42)
enc_test = MiniEncoder(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS)
dummy_ids = tf.constant([[3, 1, 4, 1]])
enc_out = enc_test(dummy_ids)
print("Encoder output shape:", enc_out.shape)
print(f"  -> (batch=1, seq_len={SEQ_LEN}, d_model={D_MODEL})")
print("  -> One enriched vector per source token, NOT a next-token prediction")

#### Predict before you run — encoder heatmap row 0

In the decoder's causal heatmap, row 0 has exactly **1** non-zero cell (token 0 attends only to itself).

**Predict:** In the encoder heatmap, how many non-zero cells will row 0 have?

A. 1 (same as decoder — only itself)  
B. 2 (attends to immediate neighbours only)  
C. 4 (attends to all positions equally)

Write your answer, then run the encoder-vs-decoder heatmap experiment below.

In [ ]:
#  Encoder vs Decoder attention heatmap: SAME weights, ONLY the mask differs
tf.random.set_seed(7)
shared_mha = layers.MultiHeadAttention(num_heads=N_HEADS, key_dim=D_MODEL // N_HEADS)

x_demo = tf.random.normal([1, SEQ_LEN, D_MODEL])

# Encoder: bidirectional (mask=None)
_, w_encoder = shared_mha(x_demo, x_demo, return_attention_scores=True)

# Decoder: causal mask (True = attend, lower triangular)
i = tf.range(SEQ_LEN)[:, tf.newaxis]
j = tf.range(SEQ_LEN)[tf.newaxis, :]
causal_mask = tf.cast(i >= j, tf.bool)
_, w_decoder = shared_mha(x_demo, x_demo, attention_mask=causal_mask,
                           return_attention_scores=True)

w_enc_h0 = w_encoder[0, 0].numpy()
w_dec_h0 = w_decoder[0, 0].numpy()
labels = ["3", "1", "4", "1"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, data, title, cmap in [
    (axes[0], w_enc_h0, "Encoder (mask=None)\nBidirectional", "Blues"),
    (axes[1], w_dec_h0, "Decoder (causal mask)\nLower-triangle only", "Oranges"),
]:
    sns.heatmap(data, ax=ax, annot=True, fmt=".2f", cmap=cmap,
                xticklabels=labels, yticklabels=labels, linewidths=0.5,
                cbar=True, vmin=0, vmax=1, cbar_kws={"label": "attention weight"})
    ax.set_title(title, fontsize=11); ax.set_xlabel("Key position"); ax.set_ylabel("Query position")

plt.suptitle("Same MHA weights, same input — only the mask differs", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

nz_enc = int((w_enc_h0[0] > 0.01).sum())
nz_dec = int((w_dec_h0[0] > 0.01).sum())
print(f"Row 0 non-zero cells — Encoder: {nz_enc}   Decoder: {nz_dec}")
print(f"  -> Encoder row 0 attends to ALL {SEQ_LEN} positions  (answer: C)")
print(f"  -> Implementation difference: one argument — mask=None vs causal_mask.")

---

## Part 3 — The Bottleneck Problem

### Why passing encoder output as decoder initial state fails

Pre-attention seq2seq models compressed the source into a single fixed-size vector. The capacity of a $d$-dimensional vector is fixed regardless of source length. Cross-attention eliminates the bottleneck by keeping **all** $S$ source vectors simultaneously accessible.

In [ ]:
#  Bottleneck: mean-pooled encoder vector loses positional detail
tf.random.set_seed(42)
enc_probe = MiniEncoder(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS)

seq_a = tf.constant([[3, 1, 4, 1]])   # main example
seq_b = tf.constant([[9, 8, 7, 6]])   # completely different sequence

out_a = enc_probe(seq_a)   # (1, 4, 32)
out_b = enc_probe(seq_b)   # (1, 4, 32)

def cos_sim(a, b):
    a = a / (tf.norm(a) + 1e-9)
    b = b / (tf.norm(b) + 1e-9)
    return float(tf.reduce_sum(a * b))

pos_sims = [cos_sim(out_a[0, i], out_b[0, i]) for i in range(SEQ_LEN)]
pool_sim = cos_sim(tf.reduce_mean(out_a, axis=1)[0],
                   tf.reduce_mean(out_b, axis=1)[0])

print("Cosine similarity: seq_a=[3,1,4,1] vs seq_b=[9,8,7,6]")
print()
print("Per-position encoder vectors:")
for i, s in enumerate(pos_sims):
    bar = "#" * int(abs(s) * 20)
    tag = "distinct" if abs(s) < 0.7 else "similar"
    print(f"  position {i}: {s:+.4f}  {bar}  ({tag})")
print()
print(f"Mean-pooled vector: {pool_sim:+.4f}")
print()
print("  -> Cross-attention keeps all", SEQ_LEN, "source vectors alive.")
print("  -> The decoder queries exactly the positions it needs at each step.")

---

## Part 4 — Cross-Attention: The Bridge

### Two problems with the naive alternatives

Problem 1 — The bottleneck (recap): a single pooled vector loses per-position info.  
Problem 2 — Concatenation into self-attention breaks: the causal mask would block source tokens, and source/target lengths grow differently.

Cross-attention solves both by keeping Q and K/V as two genuinely separate streams:

$$Q = \text{decoder state} \cdot W_Q \qquad K = \text{encoder output} \cdot W_K \qquad V = \text{encoder output} \cdot W_V$$

The score matrix is $(T_{\text{tgt}} \times S_{\text{src}})$ — asymmetric by design. No mask is applied to the encoder dimension.

In [ ]:
#  Cross-attention using tf.keras.layers.MultiHeadAttention
#  Q from decoder, K/V from encoder — two separate input streams

class CrossAttentionLayer(tf.keras.layers.Layer):
    """
    Cross-attention: Q from decoder_x, K/V from encoder_out.
    forward(decoder_x, encoder_out) ->  output (B, T, d_model)
    """
    def __init__(self, d_model, n_heads, **kwargs):
        super().__init__(**kwargs)
        self.mha = layers.MultiHeadAttention(num_heads=n_heads,
                                              key_dim=d_model // n_heads,
                                              dropout=0.0)
    def call(self, decoder_x, encoder_out, training=False, return_scores=False):
        # query = decoder, value = encoder, key = encoder
        # No mask on encoder side — decoder can attend to ANY source position
        out, attn_w = self.mha(
            query=decoder_x, value=encoder_out, key=encoder_out,
            return_attention_scores=True, training=training
        )
        if return_scores:
            return out, attn_w
        return out

# Demo: 1 decoder query attending to 4 encoder positions
tf.random.set_seed(42)
ca_demo = CrossAttentionLayer(D_MODEL, N_HEADS)
enc_dummy = tf.random.normal([1, SEQ_LEN, D_MODEL])   # 4 source tokens
dec_dummy = tf.random.normal([1, 1, D_MODEL])          # 1 decoder step

ca_out, ca_w = ca_demo(dec_dummy, enc_dummy, return_scores=True)
print("Cross-attention shapes:")
print(f"  Decoder Q   : {dec_dummy.shape}  (1 decoder token)")
print(f"  Encoder K/V : {enc_dummy.shape}  ({SEQ_LEN} source tokens)")
print(f"  Score matrix: {ca_w.shape}")
print(f"  Output      : {ca_out.shape}")
print()
w_h0 = ca_w[0, 0, 0].numpy()
print("Attention weights over source positions (head 0, 1 query step):")
for pos, w in enumerate(w_h0):
    bar = "#" * int(w * 30)
    print(f"  src[{pos}]: {w:.3f}  {bar}")

#### Your turn — cross-attention shape when T ≠ S?

The demo above ran exactly 1 decoder step against 4 source positions. The score matrix is `(T, S)` — asymmetric.

**Predict:** if the decoder has **6** steps but the source still has 4 positions, what shape will the cross-attention score matrix be — `(6, 4)`, `(4, 6)`, or `(6, 6)`?

Change `ex_t_steps` below and re-run to check.

In [ ]:
#  EXERCISE — cross-attention shape when T != S
ex_t_steps = 6   # CHANGE: try 1, 4, 6, 10 — source length stays fixed at SEQ_LEN

tf.random.set_seed(42)
ca_ex = CrossAttentionLayer(D_MODEL, N_HEADS)
enc_ex = tf.random.normal([1, SEQ_LEN, D_MODEL])    # source: fixed
dec_ex = tf.random.normal([1, ex_t_steps, D_MODEL])  # target: varies

out_ex, w_ex = ca_ex(dec_ex, enc_ex, return_scores=True)
print(f"Decoder steps (T) = {ex_t_steps},  Source positions (S) = {SEQ_LEN}")
print(f"  Score matrix shape : {w_ex.shape}  -> (batch, n_heads, T={ex_t_steps}, S={SEQ_LEN})")
print(f"  Output shape       : {out_ex.shape}")
print()
if w_ex.shape[-2] == ex_t_steps and w_ex.shape[-1] == SEQ_LEN:
    print(f"  -> Confirmed: cross-attention is asymmetric ({ex_t_steps} x {SEQ_LEN}), never (S x S).")

#### Predict before you run — what will the trained cross-attention map look like?

For `[3, 1, 4, 1]` → `[1, 4, 1, 3]`:
- Decoder step 0 must output `1` — which is at **source position 3**
- Decoder step 1 must output `4` — which is at **source position 2**
- Decoder step 2 must output `1` — which is at **source position 1**  
- Decoder step 3 must output `3` — which is at **source position 0**

**Predict:** the trained cross-attention map will look like:

A. The identity matrix (step 0 attends to src 0, etc.)  
B. The anti-diagonal (step 0 attends to src 3, step 1 to src 2, etc.)  
C. Uniform attention

Write your answer. Part 6 reveals it.

---

## Part 5 — Full Encoder-Decoder: Training

### Wiring encoder + cross-attention + decoder

The complete model stacks three components:
1. **Encoder** — bidirectional blocks; produces source map $(B, S, D)$
2. **Decoder** — causal self-attention + cross-attention at every block
3. **Language model head** — linear projection from $D$ to vocabulary size

Teacher forcing: decoder input = `[BOS] + target[:-1]`, decoder target = `target + [EOS]`.

In [ ]:
#  DecoderBlock: causal self-attn + cross-attn + FFN
class DecoderBlock(tf.keras.layers.Layer):
    """Decoder layer: causal self-attn + cross-attn + FFN (Pre-LN)."""

    def __init__(self, d_model, n_heads, d_ff, **kwargs):
        super().__init__(**kwargs)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.self_attn = layers.MultiHeadAttention(num_heads=n_heads,
                                                    key_dim=d_model // n_heads,
                                                    dropout=0.0)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.cross_attn = layers.MultiHeadAttention(num_heads=n_heads,
                                                     key_dim=d_model // n_heads,
                                                     dropout=0.0)
        self.norm3 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn = FeedForward(d_model, d_ff)

    def call(self, x, encoder_out, causal_mask=None, training=False,
             return_cross_attn=False):
        # 1. Causal self-attention (decoder reads its own past)
        normed1 = self.norm1(x)
        sa_out = self.self_attn(normed1, normed1, attention_mask=causal_mask,
                                training=training)
        x = x + sa_out
        # 2. Cross-attention: Q from decoder, K/V from encoder (no mask)
        normed2 = self.norm2(x)
        if return_cross_attn:
            ca_out, ca_w = self.cross_attn(normed2, encoder_out, encoder_out,
                                            return_attention_scores=True,
                                            training=training)
        else:
            ca_out = self.cross_attn(normed2, encoder_out, encoder_out,
                                      training=training)
            ca_w = None
        x = x + ca_out
        # 3. Feed-forward
        x = x + self.ffn(self.norm3(x), training=training)
        return (x, ca_w) if return_cross_attn else x

In [ ]:
#  EncoderDecoder: full seq2seq model
class EncoderDecoder(tf.keras.Model):
    """
    Full encoder-decoder transformer.
    encode(src_ids)       -> encoder_out (B, S, d_model)
    decode(tgt_ids, ...)  -> logits (B, T, vocab_size), ca_w
    forward call(src_ids, tgt_ids) -> logits (B, T, vocab_size), ca_w
    """

    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers,
                 max_seq=64, **kwargs):
        super().__init__(**kwargs)
        self.encoder = MiniEncoder(vocab_size, d_model, n_heads, d_ff, n_layers, max_seq)
        self.dec_emb = layers.Embedding(vocab_size, d_model, mask_zero=False)
        self.dec_pe = sinusoidal_pe(max_seq, d_model)
        self.dec_blocks = [DecoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        self.dec_norm = layers.LayerNormalization(epsilon=1e-6)
        self.lm_head = layers.Dense(vocab_size, use_bias=False)

    def encode(self, src_ids, training=False):
        return self.encoder(src_ids, training=training)

    def decode(self, tgt_ids, encoder_out, causal_mask, training=False,
               return_cross_attn=False):
        T = tf.shape(tgt_ids)[1]
        x = self.dec_emb(tgt_ids) + self.dec_pe[:T]
        ca_w = None
        for block in self.dec_blocks:
            if return_cross_attn:
                x, ca_w = block(x, encoder_out, causal_mask=causal_mask,
                                training=training, return_cross_attn=True)
            else:
                x = block(x, encoder_out, causal_mask=causal_mask, training=training)
        x = self.dec_norm(x)
        return self.lm_head(x), ca_w

    def call(self, src_ids, tgt_ids, training=False, return_cross_attn=False):
        encoder_out = self.encode(src_ids, training=training)
        T = tf.shape(tgt_ids)[1]
        # Causal mask: (T, T) bool — True = attend (lower triangular)
        i = tf.range(T)[:, tf.newaxis]
        j = tf.range(T)[tf.newaxis, :]
        causal_mask = tf.cast(i >= j, tf.bool)
        return self.decode(tgt_ids, encoder_out, causal_mask, training=training,
                           return_cross_attn=return_cross_attn)

# Sanity check
tf.random.set_seed(42)
model_check = EncoderDecoder(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS)
src_t = tf.constant([[3, 1, 4, 1]])
tgt_t = tf.constant([[BOS, 1, 4, 1]])
logits_check, _ = model_check(src_t, tgt_t)
n_params = sum(np.prod(v.shape) for v in model_check.trainable_variables)
print(f"EncoderDecoder: vocab={VOCAB_SIZE}, d_model={D_MODEL}, n_heads={N_HEADS}")
print(f"Total parameters: {n_params:,}")
print(f"Forward pass: src {src_t.shape}  tgt_in {tgt_t.shape}")
print(f"  -> logits {logits_check.shape}   (B, T, vocab_size)")

### Code Walkthrough: Full Encoder-Decoder, Recapped

**4 key patterns:**

**`DecoderBlock` — three sub-layers in strict order:** (1) causal self-attention on the decoder's own past; (2) cross-attention querying the encoder; (3) feed-forward. The ordering matters: self-attention first lets the decoder incorporate its own generated context before querying the encoder.

**`EncoderDecoder.encode` runs once, reused by every decoder block:** `self.encoder(src_ids)` returns `(B, S, d_model)`. This tensor is passed to every `DecoderBlock` inside `decode()`. Generating 100 output tokens costs just 1 encoder pass + 100 decoder passes.

**Causal mask: `tf.cast(i >= j, tf.bool)` — future tokens blocked:** `i >= j` is True for the lower triangle (position $i$ can attend to $j \leq i$). Only positions $j > i$ are blocked.

**`lm_head` — projects hidden state to vocabulary:** `layers.Dense(vocab_size, use_bias=False)` maps each decoder position to `vocab_size` unnormalised logits. Loss is `SparseCategoricalCrossentropy(from_logits=True)`.

In [ ]:
#  Training loop: teacher-forced seq2seq with tf.GradientTape
def build_dataset(srcs, tgts):
    """Pack lists of int sequences into TF tensors."""
    src_t = tf.constant(srcs, dtype=tf.int32)
    tgt_in = tf.concat([
        tf.fill([len(tgts), 1], BOS),
        tf.constant(tgts, dtype=tf.int32),
    ], axis=1)
    tgt_out = tf.concat([
        tf.constant(tgts, dtype=tf.int32),
        tf.fill([len(tgts), 1], EOS),
    ], axis=1)
    return src_t, tgt_in, tgt_out

src_train, tgt_in_train, tgt_out_train = build_dataset(train_srcs, train_tgts)
src_val,   tgt_in_val,   tgt_out_val   = build_dataset(val_srcs,   val_tgts)

tf.random.set_seed(42)
model = EncoderDecoder(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS)
optimizer = tf.keras.optimizers.Adam(learning_rate=3e-3)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

EPOCHS = 30
BATCH_SIZE = 64
train_losses, val_losses = [], []

n_train = src_train.shape[0]

for epoch in range(1, EPOCHS + 1):
    # Mini-batch training
    ep_loss = 0.0; n_batches = 0
    for start in range(0, n_train, BATCH_SIZE):
        end = start + BATCH_SIZE
        src_b  = src_train[start:end]
        tin_b  = tgt_in_train[start:end]
        tout_b = tgt_out_train[start:end]
        with tf.GradientTape() as tape:
            logits, _ = model(src_b, tin_b, training=True)
            loss = loss_fn(tout_b, logits)
        grads = tape.gradient(loss, model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        ep_loss += loss.numpy(); n_batches += 1
    ep_loss /= n_batches
    train_losses.append(ep_loss)
    # Validation
    val_logits, _ = model(src_val, tgt_in_val, training=False)
    v_loss = loss_fn(tgt_out_val, val_logits).numpy()
    val_losses.append(v_loss)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}  train={ep_loss:.4f}  val={v_loss:.4f}")

print()
print(f"Final train loss : {train_losses[-1]:.4f}")
print(f"Final val   loss : {val_losses[-1]:.4f}")
print("  -> Random baseline loss: ~2.56 (log(13), uniform over 13 tokens)")

#### Predict before you run — what accuracy will the trained model achieve?

We trained for 30 epochs on 2,000 reversal examples with `d_model=32`.

**Predict:** The validation sequence accuracy (all 4 digits must be correct) will be:

A. Below 50% — the model barely learns  
B. 50-85% — partial learning, many errors  
C. Above 90% — the model has cracked the reversal pattern

In [ ]:
#  Training loss curve + validation sequence accuracy
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, EPOCHS + 1), train_losses, 'b-o', ms=3, label='Train loss')
ax.plot(range(1, EPOCHS + 1), val_losses,   'r-o', ms=3, label='Val   loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-entropy loss')
ax.set_title('EncoderDecoder training curve — sequence reversal task')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Sequence-level accuracy: all SEQ_LEN tokens must be correct
val_logits, _ = model(src_val, tgt_in_val, training=False)
preds = tf.argmax(val_logits, axis=-1, output_type=tf.int32)[:, :SEQ_LEN]
gold  = tgt_out_val[:, :SEQ_LEN]
correct = tf.reduce_sum(tf.cast(
    tf.reduce_all(preds == gold, axis=1), tf.float32)).numpy()
accuracy = correct / src_val.shape[0]

print(f"Validation sequence accuracy: {accuracy:.1%}  ({int(correct)}/{src_val.shape[0]} fully correct)")
if accuracy > 0.90:
    print("  -> Excellent: the model has learned the reversal pattern.")
elif accuracy > 0.70:
    print("  -> Good: most sequences correct; a few more epochs would help.")
else:
    print("  -> Still converging — try more epochs or a larger model.")
print()
print("  -> Random baseline: (1/10)^4 = 0.01% (guessing each digit independently)")

---

## Part 6 — The Cross-Attention Map

### Proving the decoder learned reversal through attention routing

For perfect reversal of `[3, 1, 4, 1]` to `[1, 4, 1, 3]`:

| Decoder step | Must output | Source position to attend to |
|---|---|---|
| 0 | 1 | 3 (last) |
| 1 | 4 | 2 |
| 2 | 1 | 1 |
| 3 | 3 | 0 (first) |

If the model learned this, the cross-attention map should be the **anti-diagonal** — proof that architecture, not memorisation, solved the task.

In [ ]:
#  Cross-attention heatmap: does decoder step i attend to source position S-1-i?
ex_src = tf.constant([[3, 1, 4, 1]])
ex_tgt_in = tf.constant([[BOS, 1, 4, 1]])

logits_ex, ca_w_ex = model(ex_src, ex_tgt_in, training=False, return_cross_attn=True)

# ca_w_ex: (1, n_heads, T, S)
ca_avg = tf.reduce_mean(ca_w_ex[0], axis=0).numpy()   # (T, S) avg over heads
ca_display = ca_avg[:SEQ_LEN, :]                        # rows 0..3 (generating steps)

src_labels = ["3", "1", "4", "1"]
tgt_labels = ["->1 (step 0)", "->4 (step 1)", "->1 (step 2)", "->3 (step 3)"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
sns.heatmap(ca_display, ax=ax, annot=True, fmt=".2f", cmap="YlOrRd",
            xticklabels=src_labels, yticklabels=tgt_labels,
            linewidths=0.5, vmin=0, vmax=1)
ax.set_title("Cross-attention (avg all heads)\nDecoder step vs. Source position", fontsize=10)
ax.set_xlabel("Source position (key)"); ax.set_ylabel("Decoder step (query)")

ax2 = axes[1]
ca_h0 = ca_w_ex[0, 0, :SEQ_LEN, :].numpy()
sns.heatmap(ca_h0, ax=ax2, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=src_labels, yticklabels=tgt_labels,
            linewidths=0.5, vmin=0, vmax=1)
ax2.set_title("Cross-attention (head 0 only)", fontsize=10)
ax2.set_xlabel("Source position (key)"); ax2.set_ylabel("Decoder step (query)")

plt.suptitle("Cross-attention map: [3,1,4,1] -> [1,4,1,3]", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

anti_diag = sum(ca_display[i, SEQ_LEN - 1 - i] for i in range(SEQ_LEN)) / SEQ_LEN
print(f"Mean attention weight on anti-diagonal positions: {anti_diag:.3f}")
if anti_diag > 0.5:
    print("  -> Strong anti-diagonal pattern confirmed!")
    print("     Answer to the Part 4 prediction: B (anti-diagonal).")
else:
    print("  -> Pattern more diffuse. Try training for more epochs.")
print()
preds_ex = tf.argmax(logits_ex, axis=-1)[0, :SEQ_LEN].numpy().tolist()
gold_ex = [1, 4, 1, 3]
print(f"Model prediction (greedy): {preds_ex}")
print(f"Gold target               : {gold_ex}")
print(f"  -> {'Correct!' if preds_ex == gold_ex else 'Incorrect — try more training epochs.'}")

---

## Part 6a — Free-Running Decoding (Teacher Forcing vs Autoregressive)

During training we used **teacher forcing** — feed the ground-truth previous token at every step. At inference we must use the model's own previous prediction. This creates **exposure bias**: the model was never trained to recover from its own errors.

Let's measure this directly by comparing teacher-forced accuracy against autoregressive greedy decoding.

In [ ]:
#  Autoregressive greedy decoding (no teacher forcing)
def greedy_decode(model, src_ids_list, max_len=SEQ_LEN + 2):
    """Auto-regressive greedy decoding — no teacher forcing."""
    src = tf.constant([src_ids_list])
    encoder_out = model.encode(src, training=False)
    dec_seq = [BOS]
    for _ in range(max_len):
        tgt_in = tf.constant([dec_seq])
        T = len(dec_seq)
        i = tf.range(T)[:, tf.newaxis]
        j = tf.range(T)[tf.newaxis, :]
        causal_mask = tf.cast(i >= j, tf.bool)
        logits, _ = model.decode(tgt_in, encoder_out, causal_mask, training=False)
        next_id = int(tf.argmax(logits[0, -1], axis=-1).numpy())
        if next_id == EOS:
            break
        dec_seq.append(next_id)
    return dec_seq[1:]   # strip BOS

# Measure autoregressive vs teacher-forced accuracy on validation set
ar_correct = 0
for src_seq, tgt_seq in zip(val_srcs[:100], val_tgts[:100]):
    pred = greedy_decode(model, src_seq)
    if pred == tgt_seq:
        ar_correct += 1

ar_accuracy = ar_correct / 100
print(f"Autoregressive (greedy) accuracy: {ar_accuracy:.1%}  on first 100 val examples")
print(f"Teacher-forced accuracy above   : {accuracy:.1%}")
print()
print("  -> Teacher forcing sees the GOLD input at each step (easier to train)")
print("  -> Autoregressive uses its OWN previous output (harder at inference)")
print("  -> The gap is exposure bias — larger with more errors cascading")
print()
print("Example predictions:")
for src_seq, tgt_seq in zip(val_srcs[:5], val_tgts[:5]):
    pred = greedy_decode(model, src_seq)
    tag = "OK" if pred == tgt_seq else "WRONG"
    print(f"  src={src_seq}  gold={tgt_seq}  pred={pred}  [{tag}]")

---

## Part 7 — Toy to Real: T5 / BART Parameter Mapping

Every component you built is present in T5 and BART — same architecture, wider vectors.

| Component | This notebook | T5-base | BART-base |
|---|---|---|---|
| `d_model` | 32 | 512 | 768 |
| `n_heads` | 4 | 8 | 12 |
| `d_head` | 8 | 64 | 64 |
| `d_ff` | 64 | 2048 | 3072 |
| `n_layers` (enc+dec) | 2+2 | 6+6 | 6+6 |
| Vocabulary | 13 | 32,128 | 50,265 |
| Total params | ~15K | ~220M | ~139M |

T5 and BART also add:
- **Relative positional bias** (T5) or **learned PE** (BART) instead of sinusoidal
- **Weight tying** between embedding matrix and lm_head projection  
- **Dropout** throughout  
- **Beam search** at inference (instead of greedy)

In [ ]:
#  Load a real T5 model (tiny variant) with HuggingFace TF
try:
    from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM
    print("Loading t5-small (~60MB)...")
    t5_tok = AutoTokenizer.from_pretrained("t5-small")
    t5 = TFAutoModelForSeq2SeqLM.from_pretrained("t5-small")
    total_t5 = sum(np.prod(v.shape) for v in t5.trainable_variables)
    print(f"  T5-small total parameters: {total_t5:,}")
    print()
    # Translation demo
    text = "translate English to French: The cat sat on the mat."
    inputs = t5_tok(text, return_tensors="tf")
    out = t5.generate(inputs["input_ids"], max_length=30)
    translated = t5_tok.decode(out[0], skip_special_tokens=True)
    print(f"Input    : {text!r}")
    print(f"T5 output: {translated!r}")
except Exception as exc:
    print(f"T5 load failed (network/disk): {exc}")
    print("  -> In an offline environment, inspect the architecture instead:")
    our_params = sum(np.prod(v.shape) for v in model.trainable_variables)
    print(f"  -> Our toy model: {our_params:,} params | T5-small: ~60M | T5-base: ~220M")
    print("  -> Same components: Embedding, PE, TransformerBlock, CrossAttention, lm_head")

---

## What This Notebook Covered

| Topic | Built from scratch | Used Keras built-in |
|---|---|---|
| Sinusoidal PE | `tf.math.sin/cos` arithmetic | — |
| Bidirectional encoder | `TransformerBlock` with `mask=None` | `layers.MultiHeadAttention` |
| Cross-attention | Q from decoder, K/V from encoder | `layers.MultiHeadAttention` |
| Encoder-decoder contract | `MiniEncoder` + `DecoderBlock` | — |
| Teacher-forced training | `tf.GradientTape`, `tf.clip_by_global_norm` | `SparseCategoricalCrossentropy` |
| Cross-attention map | Heatmap + anti-diagonal verification | — |
| Autoregressive decoding | Greedy loop | — |
| Real model bridge | — | `TFAutoModelForSeq2SeqLM` (T5) |

**Scope note:** the bottleneck comparison, exposure-bias measurement, and cross-attention anti-diagonal proof are the three load-bearing demonstrations of this chapter. Weight tying, beam search, and relative PE are named but not built here — see the curriculum map in `notes/`.